In [ ]:
from collections import defaultdict
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
from context_flux_no.metrics import relative_L2_error, relative_L_infty_error
from context_flux_no.training.io import load_model
from einops import rearrange
from tqdm import tqdm


datadir = Path("../../data")
checkpoint_dir = Path("../../checkpoints/cubicflux_1d")

jax.config.update("jax_default_device", jax.devices("gpu")[2])

In [ ]:
@eqx.filter_jit
def compute_metrics(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]

    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]

    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }

In [ ]:
model_paths = {
    5: [
        # "seed=0/",
        "seed=10/26-07-25-03_23_49",
        "seed=20/26-07-25-14_07_23",
    ],
    10: [
        # "seed=0/26-04-16-00_31_47",
        "seed=10/26-07-25-07_25_05",
        "seed=20/26-07-25-15_45_14",
    ],
    20: [
        "seed=0/26-04-16-00_31_47",
        "seed=10/26-04-17-07_57_49",
        "seed=20/26-04-17-10_50_08",
    ],
    40: [
        # "seed=0/26-04-16-00_31_47",
        "seed=10/26-07-25-10_56_49",
        "seed=20/26-07-25-16_04_56",
    ],
}

In [ ]:
dataset_test = (
    xr.open_dataset(
        datadir
        / "datasets/cubic_no_source/data/test/cubic_no_source_large_test_seed=10.hdf5",
        engine="h5netcdf",
        chunks={},
    )
    .isel(t=slice(None, None, 10))
    .isel({"t": slice(0, 99)})
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = rearrange(dataset_test["values"].values, "pde ic ... -> (pde ic) ...")
segments = np.lib.stride_tricks.sliding_window_view(values, 60, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results_dict = defaultdict(list)
for ctx_length, paths in model_paths.items():
    for p in tqdm(paths):
        model = load_model(checkpoint_dir / "HyperFluxFNOLocal/OneStepLoss" / p)
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(
                lambda u: compute_metrics(model, u, (dt, dx), context_length=ctx_length)
            ),
            segments[:, : (ctx_length + 20)],
            batch_size=6000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[ctx_length].append(jax.tree.map(jnp.mean, results))

In [ ]:
results_dict

In [ ]:
for ctx_length, res in results_dict.items():
    if ctx_length == 20:
        res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    else:
        res_ = jax.tree.transpose(jax.tree.structure(["*"] * 2), None, res)
    stats = jax.tree.map(
        lambda list_: {
            "mean": jnp.mean(jnp.asarray(list_)),
            "std": jnp.std(jnp.asarray(list_)),
        },
        res_,
        is_leaf=lambda x: isinstance(x, list),
    )
    for k, v in stats.items():
        print(ctx_length, f"{k}: {v['mean']:.2e}" + "±" + f"{v['std']:.2e}")